# TMDB Movie Data Analysis with Spark

Pipeline: fetch movie data from the TMDB API → clean it with PySpark → calculate KPIs → visualise.
Helper functions live in `src/`; this notebook runs them in order and shows the results.

In [1]:
import logging
import os
import sys
from pathlib import Path

# Make the project root importable, so "from src import ..." works from /notebooks
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Show log messages in the notebook output (stdout = normal, not red, output)
logging.basicConfig(level=logging.INFO, stream=sys.stdout,
                    format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("py4j").setLevel(logging.WARNING)   # hide Spark's internal connection messages
logger = logging.getLogger("notebook")

from src import config
from src.fetch import fetch_all_movies, save_raw_json

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\User\Desktop\LABS_Docs\tmdb-spark-analysis


## 1. Fetch movie data from the TMDB API
Raw responses (including cast and crew via `append_to_response=credits`) are saved to
`data/raw/movies.json`, so the API is only called when needed.

In [2]:
FETCH_FROM_API = True   # True = call TMDB now; False = reuse data/raw/movies.json

if FETCH_FROM_API or not config.RAW_MOVIES_FILE.exists():
    save_raw_json(fetch_all_movies())
else:
    logger.info("Using saved raw data: %s", config.RAW_MOVIES_FILE)

2026-11-03 16:20:41,770 WARNING src.fetch: Movie 0: not found (404) - skipped
2026-11-03 16:20:41,977 INFO src.fetch: Movie 299534: fetched 'Avengers: Endgame'
2026-11-03 16:20:42,223 INFO src.fetch: Movie 19995: fetched 'Avatar'
2026-11-03 16:20:42,696 INFO src.fetch: Movie 140607: fetched 'Star Wars: The Force Awakens'
2026-11-03 16:20:42,910 INFO src.fetch: Movie 299536: fetched 'Avengers: Infinity War'
2026-11-03 16:20:43,088 INFO src.fetch: Movie 597: fetched 'Titanic'
2026-11-03 16:20:43,383 INFO src.fetch: Movie 135397: fetched 'Jurassic World'
2026-11-03 16:20:43,537 INFO src.fetch: Movie 420818: fetched 'The Lion King'
2026-11-03 16:20:43,737 INFO src.fetch: Movie 24428: fetched 'The Avengers'
2026-11-03 16:20:43,899 INFO src.fetch: Movie 168259: fetched 'Furious 7'
2026-11-03 16:20:44,090 INFO src.fetch: Movie 99861: fetched 'Avengers: Age of Ultron'
2026-11-03 16:20:44,282 INFO src.fetch: Movie 284054: fetched 'Black Panther'
2026-11-03 16:20:44,450 INFO src.fetch: Movie 124

In [3]:
from pyspark.sql import SparkSession

# Spark's Python workers must use the same Python as this notebook (not the Store "python3")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("tmdb-analysis")
    .master("local[*]")                              # run on this PC, all CPU cores
    .config("spark.sql.shuffle.partitions", "4")     # default is 200: far too many for 18 rows
    .config("spark.sql.session.timeZone", "UTC")     # consistent dates on any PC
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")              # hide INFO/WARN noise in the notebook

print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark 4.3.0 | shuffle partitions: 4


In [4]:
raw_df = spark.read.json(config.RAW_MOVIES_FILE.as_posix(), multiLine=True)

print(f"Rows: {raw_df.count()} | Columns: {len(raw_df.columns)}")
raw_df.select("id", "title", "budget", "revenue").show(truncate=False)

Rows: 18 | Columns: 28
+------+--------------------------------------------+---------+----------+
|id    |title                                       |budget   |revenue   |
+------+--------------------------------------------+---------+----------+
|299534|Avengers: Endgame                           |356000000|2942186933|
|19995 |Avatar                                      |237000000|2923706026|
|140607|Star Wars: The Force Awakens                |245000000|2068223624|
|299536|Avengers: Infinity War                      |300000000|2052415039|
|597   |Titanic                                     |200000000|2264162353|
|135397|Jurassic World                              |150000000|1671537444|
|420818|The Lion King                               |260000000|1662020819|
|24428 |The Avengers                                |220000000|1518815515|
|168259|Furious 7                                   |190000000|1515400000|
|99861 |Avengers: Age of Ultron                     |235000000|1405403694|
|2